# SQL for Data Platforms
## Part 1: Fundamentals — DDL, DML & DQL

**Engine:** Python `sqlite3` (standard library) + `pandas` — in-memory, no server required.

This is Part 1 of the *SQL for Every Data Platform* series. It covers the three pillars of SQL:
- **DDL** (Data Definition Language) — `CREATE`, `ALTER`, `DROP`: defines schema structure
- **DML** (Data Manipulation Language) — `INSERT`, `UPDATE`, `DELETE`: writes data into that structure
- **DQL** (Data Query Language) — `SELECT`: retrieves it back out

Every later module in this series builds on the schema and data created here.

## Setup — the shared "Store" dataset

This cell creates the schema fresh, in-memory, via `sqlite3` — no server required. Later modules reuse this exact schema.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.execute("PRAGMA foreign_keys = ON")
conn.commit()

def sql(query):
    """Execute a SELECT query and return results as a pandas DataFrame."""
    return pd.read_sql_query(query, conn)

def execute(statement):
    """Execute a DDL or DML statement (CREATE, INSERT, UPDATE, DELETE, etc.)."""
    conn.execute(statement)
    conn.commit()

execute("""
CREATE TABLE customers (
    id               INTEGER  PRIMARY KEY,
    name             VARCHAR(100) NOT NULL,
    email            VARCHAR(100) UNIQUE NOT NULL,
    city             VARCHAR(50),
    membership_level VARCHAR(10)  NOT NULL DEFAULT 'basic'
                         CHECK (membership_level IN ('basic', 'premium', 'vip')),
    created_date     DATE NOT NULL DEFAULT (date('now')),
    phone            VARCHAR(20)
)
""")

execute("""
CREATE TABLE products (
    id             INTEGER PRIMARY KEY,
    name           VARCHAR(100) NOT NULL,
    category       VARCHAR(50)  NOT NULL,
    price          DECIMAL(10,2) NOT NULL CHECK (price >= 0),
    stock_quantity INTEGER       NOT NULL DEFAULT 0 CHECK (stock_quantity >= 0)
)
""")

execute("""
CREATE TABLE employees (
    id         INTEGER PRIMARY KEY,
    name       VARCHAR(100) NOT NULL,
    department VARCHAR(50)  NOT NULL,
    hire_date  DATE         NOT NULL,
    salary     DECIMAL(10,2) NOT NULL CHECK (salary > 0)
)
""")

execute("""
CREATE TABLE orders (
    id          INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date  DATE    NOT NULL DEFAULT (date('now')),
    status      VARCHAR(20) NOT NULL DEFAULT 'pending'
                    CHECK (status IN ('pending','shipped','delivered','cancelled')),
    FOREIGN KEY (customer_id) REFERENCES customers(id)
)
""")

execute("""
CREATE TABLE order_items (
    id         INTEGER PRIMARY KEY,
    order_id   INTEGER       NOT NULL,
    product_id INTEGER       NOT NULL,
    quantity   INTEGER       NOT NULL CHECK (quantity > 0),
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price >= 0),
    FOREIGN KEY (order_id)   REFERENCES orders(id),
    FOREIGN KEY (product_id) REFERENCES products(id)
)
""")

conn.executemany("""
INSERT INTO customers (id, name, email, city, membership_level, created_date)
VALUES (?, ?, ?, ?, ?, ?)
""", [
    (1,  'Alice Martin',   'alice.martin@gmail.com',     'Toronto',   'premium', '2022-03-15'),
    (2,  'Bob Chen',       'bob.chen@yahoo.com',         'Vancouver', 'basic',   '2022-05-20'),
    (3,  'Carol White',    'carol.white@gmail.com',       'Montreal',  'vip',     '2022-07-10'),
    (4,  'David Lee',      'david.lee@hotmail.com',       'Calgary',   'basic',   '2022-09-01'),
    (5,  'Emma Davis',     'emma.davis@gmail.com',        'Ottawa',    'premium', '2022-11-15'),
    (6,  'Frank Torres',   'frank.torres@outlook.com',    'Toronto',   'basic',   '2023-01-08'),
    (7,  'Grace Kim',      'grace.kim@gmail.com',         'Vancouver', 'vip',     '2023-02-20'),
    (8,  'Henry Brown',    'henry.brown@yahoo.com',       'Montreal',  'basic',   '2023-03-12'),
    (9,  'Isabel Santos',  'isabel.santos@gmail.com',     'Toronto',   'premium', '2023-04-05'),
    (10, 'James Wilson',   'james.wilson@hotmail.com',    'Calgary',   'basic',   '2023-05-18'),
    (11, 'Karen Taylor',   'karen.taylor@gmail.com',      'Ottawa',    'premium', '2023-06-22'),
    (12, 'Liam Johnson',   'liam.johnson@outlook.com',    'Toronto',   'vip',     '2023-07-30'),
    (13, 'Maria Garcia',   'maria.garcia@gmail.com',      'Montreal',  'basic',   '2023-08-14'),
    (14, 'Nathan Park',    'nathan.park@yahoo.com',       'Vancouver', 'premium', '2023-09-09'),
    (15, 'Olivia Patel',   'olivia.patel@gmail.com',      'Toronto',   'basic',   '2023-10-01'),
    (16, 'Patrick Murphy', 'patrick.murphy@hotmail.com',  'Calgary',   'basic',   '2023-11-11'),
    (17, 'Quinn Adams',    'quinn.adams@gmail.com',       'Ottawa',    'premium', '2023-12-05'),
    (18, 'Rachel Scott',   'rachel.scott@outlook.com',    'Toronto',   'basic',   '2024-01-18'),
    (19, 'Samuel Nguyen',  'samuel.nguyen@gmail.com',     'Vancouver', 'vip',     '2024-02-28'),
    (20, 'Tanya Roberts',  'tanya.roberts@yahoo.com',     'Montreal',  'basic',   '2024-03-15'),
    (21, 'Umar Ali',       'umar.ali@gmail.com',          'Toronto',   'premium', '2024-04-20'),
    (22, 'Vera Kozlov',    'vera.kozlov@hotmail.com',     'Calgary',   'basic',   '2024-05-30'),
    (23, 'Walter Bell',    'walter.bell@gmail.com',       'Ottawa',    'basic',   '2024-07-10'),
    (24, 'Xena Cruz',      'xena.cruz@yahoo.com',         'Vancouver', 'basic',   '2024-09-01'),
    (25, 'Yuki Tanaka',    'yuki.tanaka@gmail.com',       'Montreal',  'basic',   '2024-11-15'),
])

conn.executemany("""
INSERT INTO products (id, name, category, price, stock_quantity)
VALUES (?, ?, ?, ?, ?)
""", [
    (1,  'Wireless Headphones',  'Electronics', 129.99, 45),
    (2,  'Laptop Stand',         'Electronics',  49.99, 80),
    (3,  'USB-C Hub',            'Electronics',  39.99, 120),
    (4,  'Smartwatch',           'Electronics', 299.99, 25),
    (5,  'Running Jacket',       'Clothing',     89.99, 60),
    (6,  'Yoga Pants',           'Clothing',     54.99, 90),
    (7,  'Wool Sweater',         'Clothing',     75.00, 40),
    (8,  'Denim Jeans',          'Clothing',     69.99, 75),
    (9,  'Organic Coffee 1kg',   'Food',         24.99, 200),
    (10, 'Green Tea 100g',       'Food',         14.99, 300),
    (11, 'Protein Bars x12',     'Food',         34.99, 150),
    (12, 'Olive Oil 500ml',      'Food',         19.99, 250),
    (13, 'Python Programming',   'Books',        44.99, 55),
    (14, 'Data Science Handbook','Books',        54.99, 40),
    (15, 'SQL for Beginners',    'Books',        29.99, 70),
    (16, 'Machine Learning A-Z', 'Books',        49.99, 35),
    (17, 'Yoga Mat',             'Sports',       49.99, 65),
    (18, 'Resistance Bands Set', 'Sports',       29.99, 100),
    (19, 'Kettlebell 16kg',      'Sports',       79.99, 30),
    (20, 'Running Shoes',        'Sports',      139.99, 20),
])

conn.executemany("""
INSERT INTO employees (id, name, department, hire_date, salary)
VALUES (?, ?, ?, ?, ?)
""", [
    (1,  'Alice Foster',   'Sales',       '2019-03-15', 68000.00),
    (2,  'Bob Martinez',   'Sales',       '2020-06-01', 72000.00),
    (3,  'Carol Hughes',   'Sales',       '2021-09-10', 72000.00),
    (4,  'David Okafor',   'Sales',       '2022-11-20', 58000.00),
    (5,  'Eva Schneider',  'Engineering', '2019-01-15', 95000.00),
    (6,  'Frank Liu',      'Engineering', '2020-04-20', 105000.00),
    (7,  'Grace Patel',    'Engineering', '2021-07-01', 112000.00),
    (8,  'Hugo Silva',     'Engineering', '2022-02-15', 95000.00),
    (9,  'Iris Nakamura',  'Engineering', '2023-08-01', 85000.00),
    (10, 'Jake Thompson',  'Marketing',   '2019-11-01', 62000.00),
    (11, 'Kim Anderson',   'Marketing',   '2021-04-15', 70000.00),
    (12, 'Leo Ferreira',   'Marketing',   '2023-01-10', 62000.00),
    (13, 'Mia Campbell',   'HR',          '2020-08-20', 55000.00),
    (14, 'Noa Rosenberg',  'HR',          '2021-12-05', 60000.00),
    (15, 'Omar Diallo',    'HR',          '2022-05-18', 55000.00),
])

conn.executemany("""
INSERT INTO orders (id, customer_id, order_date, status)
VALUES (?, ?, ?, ?)
""", [
    (1,  1,  '2023-01-20', 'delivered'), (2,  1,  '2023-04-15', 'delivered'),
    (3,  1,  '2023-09-10', 'shipped'),   (4,  3,  '2023-02-05', 'delivered'),
    (5,  3,  '2023-06-18', 'delivered'), (6,  3,  '2023-11-22', 'pending'),
    (7,  3,  '2024-03-30', 'shipped'),   (8,  7,  '2023-03-12', 'delivered'),
    (9,  7,  '2023-08-25', 'delivered'), (10, 7,  '2024-01-10', 'pending'),
    (11, 12, '2023-02-28', 'delivered'), (12, 12, '2023-07-14', 'delivered'),
    (13, 12, '2023-12-05', 'shipped'),   (14, 12, '2024-05-20', 'pending'),
    (15, 19, '2023-04-22', 'delivered'), (16, 19, '2023-10-08', 'delivered'),
    (17, 19, '2024-03-15', 'shipped'),   (18, 2,  '2023-03-01', 'delivered'),
    (19, 4,  '2023-05-17', 'delivered'), (20, 5,  '2023-07-28', 'delivered'),
    (21, 6,  '2023-01-30', 'cancelled'), (22, 8,  '2023-06-09', 'delivered'),
    (23, 9,  '2023-08-15', 'delivered'), (24, 10, '2023-09-22', 'shipped'),
    (25, 11, '2023-10-30', 'delivered'), (26, 13, '2023-11-08', 'delivered'),
    (27, 14, '2023-12-15', 'pending'),   (28, 15, '2024-01-25', 'delivered'),
    (29, 16, '2024-02-10', 'shipped'),   (30, 17, '2024-02-28', 'delivered'),
    (31, 18, '2024-03-20', 'cancelled'), (32, 20, '2024-04-05', 'delivered'),
    (33, 21, '2024-04-18', 'shipped'),   (34, 22, '2024-05-02', 'pending'),
    (35, 2,  '2024-06-10', 'delivered'), (36, 4,  '2024-07-05', 'delivered'),
    (37, 5,  '2024-08-20', 'pending'),   (38, 6,  '2024-09-15', 'shipped'),
    (39, 8,  '2024-10-01', 'delivered'), (40, 11, '2024-12-15', 'cancelled'),
])

conn.executemany("""
INSERT INTO order_items (id, order_id, product_id, quantity, unit_price)
VALUES (?, ?, ?, ?, ?)
""", [
    (1,1,1,1,129.99),(2,1,13,1,44.99),(3,2,4,1,279.99),(4,2,17,2,49.99),
    (5,3,9,3,24.99),(6,3,15,1,29.99),(7,4,5,1,89.99),(8,4,6,2,54.99),
    (9,5,2,1,49.99),(10,5,10,2,14.99),(11,6,14,1,54.99),(12,6,18,3,29.99),
    (13,7,7,1,70.00),(14,7,16,1,49.99),(15,8,3,2,39.99),(16,8,11,4,34.99),
    (17,9,20,1,139.99),(18,9,12,2,19.99),(19,10,8,1,69.99),(20,10,19,1,79.99),
    (21,11,1,1,119.99),(22,11,9,2,24.99),(23,12,4,1,299.99),(24,12,13,2,44.99),
    (25,13,6,3,54.99),(26,13,15,1,27.99),(27,14,2,2,49.99),(28,14,17,1,49.99),
    (29,15,5,1,89.99),(30,15,10,3,14.99),(31,16,20,1,129.99),(32,16,14,1,54.99),
    (33,17,7,2,75.00),(34,17,11,2,34.99),(35,18,3,1,39.99),(36,18,18,4,29.99),
    (37,19,16,1,49.99),(38,19,12,3,19.99),(39,20,1,1,129.99),(40,20,8,1,69.99),
    (41,21,4,1,299.99),(42,21,9,2,24.99),(43,22,13,2,44.99),(44,22,15,1,29.99),
    (45,23,6,1,54.99),(46,23,19,1,75.99),(47,24,2,1,49.99),(48,24,10,4,13.99),
    (49,25,5,1,85.00),(50,25,17,2,49.99),(51,26,11,3,34.99),(52,26,20,1,139.99),
    (53,27,7,1,75.00),(54,27,14,1,49.99),(55,28,3,2,39.99),(56,28,16,1,49.99),
    (57,29,8,2,69.99),(58,29,12,2,19.99),(59,30,1,1,129.99),(60,30,18,3,28.99),
    (61,31,4,1,299.99),(62,31,9,1,24.99),(63,32,13,1,44.99),(64,32,6,2,54.99),
    (65,33,2,1,49.99),(66,33,15,2,29.99),(67,34,19,1,79.99),(68,34,10,3,14.99),
    (69,35,5,1,89.99),(70,35,16,1,49.99),(71,36,20,1,139.99),(72,36,11,2,34.99),
    (73,37,7,1,75.00),(74,37,17,1,49.99),(75,38,3,3,37.99),(76,38,12,2,19.99),
    (77,39,14,1,54.99),(78,39,18,4,29.99),(79,40,1,1,129.99),(80,40,8,1,69.99),
])
conn.commit()

print("Store dataset ready:")
for table in ["customers", "products", "employees", "orders", "order_items"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<15} {n:>3} rows")

Store dataset ready:
  customers        25 rows
  products         20 rows
  employees        15 rows
  orders           40 rows
  order_items      80 rows


## Section 1 — DDL: `CREATE TABLE`

DDL statements define **structure**, not data. They are not reversible without data loss — dropping
a table drops everything in it.

The five tables above already demonstrate the core building blocks:
- **`PRIMARY KEY`** — uniquely identifies each row (`customers.id`, `orders.id`, ...)
- **`NOT NULL`** — a column that must always have a value
- **`UNIQUE`** — no two rows may share this value (`customers.email`)
- **`DEFAULT`** — a fallback value when none is supplied (`membership_level` defaults to `'basic'`)
- **`CHECK`** — a row-level constraint enforced on every insert/update (`price >= 0`)
- **`FOREIGN KEY`** — ties a column to another table's primary key, enforcing referential integrity
  (`orders.customer_id` → `customers.id`)

> **SQLite type affinity:** SQLite uses *dynamic typing with type affinity* — it stores values in
> the most natural type but respects the declared type as a hint. `INTEGER`/`TEXT`/`REAL`/`NUMERIC`
> are the four affinities. Most platforms (PostgreSQL, MySQL, cloud warehouses) enforce static
> typing instead — a `VARCHAR(50)` column will reject a number. Keep this in mind if you're used to
> SQLite's leniency and move to a stricter engine.

In [2]:
sql("SELECT name, type FROM sqlite_master WHERE type='table' ORDER BY name")

,name,type
0,customers,table
1,employees,table
2,order_items,table
3,orders,table
4,products,table


In [3]:
sql("PRAGMA table_info(customers)")

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,NaN,1
1,1,name,VARCHAR(100),1,NaN,0
2,2,email,VARCHAR(100),1,NaN,0
3,3,city,VARCHAR(50),0,NaN,0
4,4,membership_level,VARCHAR(10),1,'basic',0
5,5,created_date,DATE,1,date('now'),0
6,6,phone,VARCHAR(20),0,NaN,0


## Section 2 — DDL: `ALTER TABLE`

SQLite's `ALTER TABLE` is intentionally minimal: `ADD COLUMN`, `RENAME TABLE`, `RENAME COLUMN`
(≥3.25). **MySQL/PostgreSQL/cloud warehouses** also support `DROP COLUMN`, `MODIFY COLUMN`/`ALTER
COLUMN TYPE`, and more — SQLite's restraint is the exception, not the rule.

In [4]:
execute("ALTER TABLE orders ADD COLUMN notes TEXT")
print("notes column added to orders")
sql("PRAGMA table_info(orders)")

notes column added to orders


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,NaN,1
1,1,customer_id,INTEGER,1,NaN,0
2,2,order_date,DATE,1,date('now'),0
3,3,status,VARCHAR(20),1,'pending',0
4,4,notes,TEXT,0,NaN,0


## Section 3 — DDL: `CREATE VIEW`

A **view** is a saved query that behaves like a read-only table — the underlying rows aren't
duplicated, the query just re-runs every time the view is selected from. Views are DDL (they
define something), even though what they define is a `SELECT`.

Views are useful for two things day to day: hiding a complex JOIN behind a simple name, and giving
a consistent, reusable definition of a metric so five analysts don't each write their own slightly
different "active customer" query. **Module 9 (Performance & Tuning)** picks this up again to
contrast plain views against *materialized* views — a view re-runs its query every time; a
materialized view caches the result and trades freshness for speed.

In [5]:
execute("""
CREATE VIEW customer_order_totals AS
SELECT
    c.id            AS customer_id,
    c.name          AS customer_name,
    c.membership_level,
    COUNT(o.id)     AS order_count,
    COALESCE(SUM(oi.quantity * oi.unit_price), 0) AS lifetime_spend
FROM customers c
LEFT JOIN orders o       ON o.customer_id = c.id
LEFT JOIN order_items oi ON oi.order_id   = o.id
GROUP BY c.id, c.name, c.membership_level
""")
print("view created")

# Query the view exactly like a table
sql("SELECT * FROM customer_order_totals ORDER BY lifetime_spend DESC LIMIT 5")

view created


,customer_id,customer_name,membership_level,order_count,lifetime_spend
0,12,Liam Johnson,vip,8,902.87
1,1,Alice Martin,premium,6,659.91
2,7,Grace Kim,vip,6,549.89
3,3,Carol White,vip,8,544.89
4,19,Samuel Nguyen,vip,6,539.92


## Section 4 — DML: `INSERT`, `UPDATE`, `DELETE`

```sql
INSERT INTO table_name (col1, col2) VALUES (val1, val2);          -- add rows (see Setup cell above)

UPDATE table_name SET col1 = val1 WHERE condition;                 -- modify existing rows

DELETE FROM table_name WHERE condition;                            -- remove rows
```

> **Always include a `WHERE` clause on `UPDATE`/`DELETE`.** Omitting it touches *every row* in the
> table — Module 14 (Style & Best Practices) makes this a hard rule: preview with `SELECT` first,
> using the exact same `WHERE` you're about to run destructively.

In [6]:
# Upgrade 3 customers to 'vip' membership
execute("UPDATE customers SET membership_level = 'vip' WHERE id IN (9, 14, 21)")
sql("SELECT id, name, membership_level FROM customers WHERE id IN (9, 14, 21)")

,id,name,membership_level
0,9,Isabel Santos,vip
1,14,Nathan Park,vip
2,21,Umar Ali,vip


In [7]:
# Safe DELETE pattern: preview first, delete with the identical WHERE
sql("SELECT id, customer_id, order_date, status FROM orders WHERE status = 'cancelled'")

,id,customer_id,order_date,status
0,21,6,2023-01-30,cancelled
1,31,18,2024-03-20,cancelled
2,40,11,2024-12-15,cancelled


With `PRAGMA foreign_keys = ON` (set in the Setup cell above), the engine enforces referential
integrity: order 40 is cancelled, but `order_items` still references it — deleting it directly
fails. **Delete children before parents.**

In [8]:
try:
    execute("DELETE FROM orders WHERE status = 'cancelled' AND id = 40")
except Exception as e:
    print(f"Failed, as expected: {e}")

Failed, as expected: FOREIGN KEY constraint failed


In [9]:
# Correct order: delete the dependent order_items first, then the order itself
execute("DELETE FROM order_items WHERE order_id = 40")
execute("DELETE FROM orders WHERE status = 'cancelled' AND id = 40")
print("order 40 and its 2 line items deleted")
sql("SELECT COUNT(*) AS remaining_cancelled FROM orders WHERE status = 'cancelled'")

order 40 and its 2 line items deleted


,remaining_cancelled
0,2


## Section 5 — DQL: `SELECT`, Clause by Clause

`SELECT` is the query written most often, and its clauses always run in the same logical
order regardless of how they're typed: `FROM` (which table) → `WHERE` (which rows) →
`GROUP BY` (Part 3) → `SELECT` (which columns/expressions) → `ORDER BY` (final sort) →
`LIMIT` (cap the row count, Part 12).

In [10]:
sql("""
SELECT name, city, membership_level        -- 4. which columns
FROM customers                              -- 1. which table
WHERE membership_level = 'vip'              -- 2. which rows
ORDER BY name ASC                           -- 5. final sort
LIMIT 10                                    -- 6. cap the results
""")

,name,city,membership_level
0,Carol White,Montreal,vip
1,Grace Kim,Vancouver,vip
2,Isabel Santos,Toronto,vip
3,Liam Johnson,Toronto,vip
4,Nathan Park,Vancouver,vip
5,Samuel Nguyen,Vancouver,vip
6,Umar Ali,Toronto,vip


## Best Practices — Fundamentals

- Always add `FOREIGN KEY` constraints where a relationship exists — they're what catches an
  orphaned `order_id` before it becomes a silent bad join downstream.
- Use `CHECK` constraints for invariants you never want violated (`price >= 0`), not for business
  rules that change often — those belong in application/pipeline logic, not baked into the schema.
- Preview every `UPDATE`/`DELETE` with the identical `WHERE` as a `SELECT` first.
- Name views for what they represent (`customer_order_totals`), not how they're built — the query
  behind a view is allowed to change; the contract with whoever queries it shouldn't.

## Next

**Part 2 — Designing the Schema: Star vs. Snowflake** uses these same tables as the *transactional*
(OLTP) shape, then shows how you'd remodel this data into a dimensional (star/snowflake) schema
built for analytics.